In [5]:
from datasets import load_dataset, load_from_disk
import os

# Get absolute path to data directory (relative to this notebook's location)
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
project_root = os.path.dirname(notebook_dir) if notebook_dir.endswith("src") else os.path.dirname(os.getcwd())
# Handle case where we're running from src directory
if os.getcwd().endswith("src"):
    project_root = os.path.dirname(os.getcwd())
else:
    project_root = os.getcwd()
print ("project root: "+project_root)
data_dir = os.path.join(project_root, "ROCOv2-radiology", "data")
cache_dir = os.path.join(project_root, "src")  # Cache in src folder

print(f"Data directory: {data_dir}")
print(f"Files exist: {os.path.exists(data_dir)}")

# Check if cached version exists for faster loading
train_cache = os.path.join(cache_dir, "roco_train_full")
val_cache = os.path.join(cache_dir, "roco_validation_full")
test_cache = os.path.join(cache_dir, "roco_test_full")

if os.path.exists(train_cache):
    print("Loading from cached Arrow format (fast)...")
    train_dataset = load_from_disk(train_cache)
    val_dataset = load_from_disk(val_cache)
    test_dataset = load_from_disk(test_cache)
else:
    print("Loading from parquet files (first time)...")
    # Use absolute paths with glob pattern
    train_files = os.path.join(data_dir, "train-*.parquet")
    val_files = os.path.join(data_dir, "validation-*.parquet")
    test_files = os.path.join(data_dir, "test-*.parquet")
    
    train_dataset = load_dataset("parquet", data_files=train_files, split="train")
    val_dataset = load_dataset("parquet", data_files=val_files, split="train")
    test_dataset = load_dataset("parquet", data_files=test_files, split="train")
    
    # Cache for faster subsequent loads
    train_dataset.save_to_disk(train_cache)
    val_dataset.save_to_disk(val_cache)
    test_dataset.save_to_disk(test_cache)
    print("✓ Cached to disk")

print(f"\nDataset loaded:")
print(f"Train: {len(train_dataset):,} samples")
print(f"Validation: {len(val_dataset):,} samples")
print(f"Test: {len(test_dataset):,} samples")

project root: c:\Users\charb\OneDrive\Desktop\deep learning final project
Data directory: c:\Users\charb\OneDrive\Desktop\deep learning final project\ROCOv2-radiology\data
Files exist: True
Loading from parquet files (first time)...


Generating train split: 59962 examples [00:20, 2871.93 examples/s]

Generating train split: 9904 examples [00:04, 2435.20 examples/s]

Generating train split: 9927 examples [00:04, 2401.76 examples/s]

Saving the dataset (6/6 shards): 100%|██████████| 9904/9904 [00:09<00:00, 1084.73 examples/s]

Saving the dataset (6/6 shards): 100%|██████████| 9927/9927 [00:09<00:00, 1038.09 examples/s]

✓ Cached to disk

Dataset loaded:
Train: 59,962 samples
Validation: 9,904 samples
Test: 9,927 samples


In [6]:
# ===== From now on, use this (super fast) =====
train_dataset = load_from_disk("roco_train_full")
val_dataset = load_from_disk("roco_validation_full")
test_dataset = load_from_disk("roco_test_full")

print(f"\nLoaded instantly:")
print(f"Train: {len(train_dataset)} samples")
print(f"Validation: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")


Loaded instantly:
Train: 59962 samples
Validation: 9904 samples
Test: 9927 samples


In [7]:
train_dataset

Dataset({
    features: ['image', 'image_id', 'caption', 'cui'],
    num_rows: 59962
})

In [ ]:

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Llama-3.2-11B-Vision-Instruct",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,                           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,                  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,               # We support rank stabilized LoRA
    loftq_config = None,               # And LoftQ
    target_modules = "all-linear",    # Optional now! Can specify a list if needed
    modules_to_save=[
        "lm_head",
        "embed_tokens",
    ],
)

In [2]:
print("hi")

hi
